In [ ]:
%cd ../..

import os
import numpy as np
import random

import torch
from einops import rearrange
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from evaluation import *

random.seed(4)
np.random.seed(4)
torch.manual_seed(4)

In [ ]:
embeddings_path = "/data/work/vm/radio-foundation/embeddings/LUNA16/detection"
embeddings_path_train = os.path.join(embeddings_path, "train")
embeddings_path_val = os.path.join(embeddings_path, "val")

train_paths = [os.path.join(embeddings_path_train, p) for p in os.listdir(embeddings_path_train)]
val_paths = [os.path.join(embeddings_path_val, p) for p in os.listdir(embeddings_path_val)]

In [ ]:
class EmbeddingDatasetLocation(Dataset):
    def __init__(self, paths_list, add_noise=False, sigma=0.05, p=0.5):
        self.paths_list = paths_list
        self.add_noise = add_noise
        self.sigma = sigma
        self.p = p

    def add_embedding_noise(self, embeddings):
        if torch.rand(1).item() < self.p:
            noise = torch.randn_like(embeddings) * self.sigma
            embeddings = embeddings + noise
        return embeddings

    def __len__(self):
        return len(self.paths_list)

    def __getitem__(self, idx):
        embedding_path = self.paths_list[idx]

        aug_idx = torch.randint(0, 10, (1,)).item()

        embedding_data = torch.load(embedding_path)
        embeddings = embedding_data[f"patch_{aug_idx:02}"]
        label = embedding_data[f"pos_{aug_idx:02}"]
        
        embeddings = rearrange(embeddings, "1 y x d -> y x d")

        if self.add_noise:
            embeddings = self.add_embedding_noise(embeddings)

        return embeddings, label


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class PositionalAttentionRegressor(nn.Module):
    def __init__(self, embed_dim, n_heads=4, hidden_dim=128):
        super().__init__()
        
        self.d_model = embed_dim
        
        self.attn1 = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=n_heads, batch_first=True)
        #self.attn2 = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=n_heads, batch_first=True)
        
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
            nn.Sigmoid()
        )

    def forward(self, x):
        """
        x: (B, Nx, Ny, D)
        """
        B, Nx, Ny, D = x.shape
        N = Nx * Ny

        x = x.view(B, N, D)
        
        grid_x, grid_y = torch.meshgrid(
            torch.linspace(0, 1, Nx, device=x.device),
            torch.linspace(0, 1, Ny, device=x.device),
            indexing="ij"
        )
        pos = torch.stack([grid_x, grid_y], dim=-1).view(N, 2)

        pos_enc = F.linear(pos, torch.empty(D,2, device=x.device).uniform_(-0.1,0.1))
        pos_enc = pos_enc.unsqueeze(0).expand(B, N, D)

        q = x + pos_enc

        out, _ = self.attn1(q, x, x)
        #out, _ = self.attn2(out, x, x)

        pooled = out.mean(dim=1)
        coords = self.fc(pooled)

        return coords

In [ ]:
train_dataset = EmbeddingDatasetLocation(train_paths, add_noise=True, p=1.0, sigma=0.1)
val_dataset = EmbeddingDatasetLocation(val_paths)

In [ ]:
from sklearn.metrics import mean_absolute_error

def train_locator(
    model,
    optimizer,
    loss_fn,
    train_dataloader,
    val_dataloader,
    num_epochs,
    device,
):
    train_loss_list = []
    train_mae_list = []
    val_loss_list = []
    val_mae_list = []

    best_val_loss = float("inf")
    best_model_state = model.state_dict()

    for epoch in tqdm(range(num_epochs)):
        model.train()
        train_all_labels = []
        train_all_predictions = []
        train_loss = 0.0

        for embeddings, labels in train_dataloader:
            embeddings, labels = (
                embeddings.to(device),
                labels.to(device),
            )

            predictions = model(embeddings)
            loss = loss_fn(predictions, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_all_labels.append(labels.detach().cpu())
            train_all_predictions.append(predictions.detach().cpu())

        train_labels_cat = torch.cat(train_all_labels).numpy()
        train_predictions_cat = torch.cat(train_all_predictions).numpy()

        avg_train_loss = train_loss / len(train_dataloader)
        train_mae = mean_absolute_error(train_labels_cat, train_predictions_cat)

        train_loss_list.append(avg_train_loss)
        train_mae_list.append(train_mae)

        model.eval()
        val_all_labels = []
        val_all_predictions = []
        val_loss = 0.0

        with torch.no_grad():
            for embeddings, labels in val_dataloader:
                embeddings, labels = (
                    embeddings.to(device),
                    labels.to(device)
                )

                predictions = model(embeddings)
                loss = loss_fn(predictions, labels)

                val_loss += loss.item()
                val_all_labels.append(labels.detach().cpu())
                val_all_predictions.append(predictions.detach().cpu())

        val_labels_cat = torch.cat(val_all_labels).numpy()
        val_predictions_cat = torch.cat(val_all_predictions).numpy()

        avg_val_loss = val_loss / len(val_dataloader)
        val_mae = mean_absolute_error(val_labels_cat, val_predictions_cat)

        val_loss_list.append(avg_val_loss)
        val_mae_list.append(val_mae)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()

    return {
        "train_loss": train_loss_list,
        "train_mae": train_mae_list,
        "val_loss": val_loss_list,
        "val_mae": val_mae_list,
        "state_dict": best_model_state,
    }


In [ ]:
EMBED_DIM = 768
num_epochs = 150
batch_size = 16
learning_rate = 0.0001

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    batch_size=batch_size,
)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

device = torch.device("cuda")
model = PositionalAttentionRegressor(embed_dim=EMBED_DIM).to(device)

loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

output = train_locator(
    model,
    optimizer,
    loss_fn,
    train_dataloader,
    val_dataloader,
    num_epochs,
    device
)
model.load_state_dict(output["state_dict"])


In [ ]:
plot_train_curves(output["train_mae"], output["val_mae"], "Mean Absolute Error")